In [1]:
import os
import pandas

# Construction du csv emotyc — features binaires


In [2]:


FOLDER_EMOTION = "sentence_level_all_features"  # <-- adapte ce chemin

features_found = set()

for filename in os.listdir(FOLDER_EMOTION):
    if not filename.endswith(".txt"):
        continue
    filepath = os.path.join(FOLDER_EMOTION, filename)
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if "::" in line:
                feature_name = line.split("::")[0].strip()
                if feature_name and feature_name != "Features":
                    features_found.add(feature_name)

print(f"{len(features_found)} features uniques trouvées :\n")
for feat in sorted(features_found):
    print(f"  - {feat}")

16 features uniques trouvées :

  - avec_cat_admiration
  - avec_cat_autre
  - avec_cat_colere
  - avec_cat_embarras
  - avec_cat_fierte
  - avec_cat_joie
  - avec_cat_peur
  - avec_cat_surprise
  - avec_cat_tristesse
  - avec_emo
  - avec_emo_base
  - avec_emo_complexe
  - avec_emo_comportementale
  - avec_emo_designee
  - avec_emo_montree
  - avec_emo_suggeree


In [3]:
type(features_found)

set

In [4]:

NO_FEATURES_MESSAGE ="No features found."

ALL_FEATURES = list(features_found)

In [5]:
def extract_text(lines):
    if lines[-1]!=NO_FEATURES_MESSAGE:
     l = len(lines)
     trueline = []
     i = 4
     while True:
        if lines[i].startswith("Features::"):
            break
        trueline.append(lines[i])
        i+=1
        return " ".join(trueline).strip().split("Sentence: ")[1].strip()
    else:
     return " ".join(lines[4:-1]).strip().split("Sentence: ")[1].strip()


def extract_features(lines):
    features = {}
    for line in lines:
        line = line.strip()
        if "::" in line:
            parts = line.split("::")
            feature_name = parts[0].strip()
            if feature_name in ALL_FEATURES:
                features[feature_name] = 1
    return features

In [6]:
def extract_from_file(lines, id):

    row = {"file_id": id, "feature_found": False}
    for f in ALL_FEATURES:
        row[f] = 0

    if lines[-1].strip() == NO_FEATURES_MESSAGE:
        print("No features found for file id: ", id)
        row["text"] = ("\n").join(lines[3:-1]).split("Sentence: ")[1].strip()
    else:
        row["text"] = extract_text(lines)
        row["feature_found"] = True
        found = extract_features(lines)
        for fname, val in found.items():
            row[fname] = val

    return row

In [7]:
rows = []

for filename in os.listdir(FOLDER_EMOTION):
    if not filename.endswith(".txt"):
        continue
    file_id = filename.replace(".txt", "")
    filepath = os.path.join(FOLDER_EMOTION, filename)
    with open(filepath, 'r', encoding='utf-8') as f:
        lines = f.read().splitlines()
    # On filtre les lignes vides en fin de fichier
    while lines and lines[-1].strip() == "":
        lines.pop()
    rows.append(extract_from_file(lines, file_id))

No features found for file id:  10
No features found for file id:  104
No features found for file id:  118
No features found for file id:  12
No features found for file id:  122
No features found for file id:  13
No features found for file id:  133
No features found for file id:  141
No features found for file id:  143
No features found for file id:  147
No features found for file id:  153
No features found for file id:  16
No features found for file id:  161
No features found for file id:  162
No features found for file id:  163
No features found for file id:  178
No features found for file id:  181
No features found for file id:  186
No features found for file id:  187
No features found for file id:  190
No features found for file id:  191
No features found for file id:  192
No features found for file id:  193
No features found for file id:  194
No features found for file id:  195
No features found for file id:  198
No features found for file id:  2
No features found for file id:  20

In [8]:
print(len(rows), "rows extracted.")
df = pandas.DataFrame(rows)

793 rows extracted.


In [9]:
df.columns

Index(['file_id', 'feature_found', 'avec_cat_joie', 'avec_cat_admiration',
       'avec_cat_tristesse', 'avec_cat_surprise', 'avec_emo_comportementale',
       'avec_cat_autre', 'avec_emo_designee', 'avec_emo', 'avec_emo_base',
       'avec_emo_suggeree', 'avec_cat_embarras', 'avec_cat_colere',
       'avec_cat_fierte', 'avec_emo_complexe', 'avec_emo_montree',
       'avec_cat_peur', 'text'],
      dtype='object')

In [10]:
df = df.set_index("file_id")
df.index = pandas.to_numeric(df.index, errors='coerce')
df = df.sort_index()
df.head()

,feature_found,avec_cat_joie,avec_cat_admiration,avec_cat_tristesse,avec_cat_surprise,avec_emo_comportementale,avec_cat_autre,avec_emo_designee,avec_emo,avec_emo_base,avec_emo_suggeree,avec_cat_embarras,avec_cat_colere,avec_cat_fierte,avec_emo_complexe,avec_emo_montree,avec_cat_peur,text
file_id,,,,,,,,,,,,,,,,,,
1,True,0,0,0,1,0,0,0,1,1,1,0,0,0,0,1,1,Tous ces migrants qui envahissent nos hôpitaux!!!
2,False,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,Manquerait plus que la république tchèque prop...
3,False,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,https://t.co/7H7Rs0CkGJ
4,False,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,Les #migrants #maghrebins qui envahissent la🇫🇷...
5,True,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,"oui, mais y'a soros (ami de macron, soit dit e..."


In [11]:
df['feature_found'].value_counts()

feature_found
False    410
True     383
Name: count, dtype: int64

In [12]:
# Aperçu des features binaires pour les phrases avec au moins une feature
feature_cols = ALL_FEATURES
df[df['feature_found'] == True][['text'] + feature_cols].head(10)

,text,avec_cat_joie,avec_cat_admiration,avec_cat_tristesse,avec_cat_surprise,avec_emo_comportementale,avec_cat_autre,avec_emo_designee,avec_emo,avec_emo_base,avec_emo_suggeree,avec_cat_embarras,avec_cat_colere,avec_cat_fierte,avec_emo_complexe,avec_emo_montree,avec_cat_peur
file_id,,,,,,,,,,,,,,,,,
1,Tous ces migrants qui envahissent nos hôpitaux!!!,0,0,0,1,0,0,0,1,1,1,0,0,0,0,1,1
5,"oui, mais y'a soros (ami de macron, soit dit e...",0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
6,En plein délire parce que 3 migrants ont fabri...,0,1,0,0,0,0,0,1,1,1,0,0,0,0,0,0
8,"Les migrants envahissent la France, c’est un f...",0,0,0,0,0,0,0,1,1,1,0,1,0,0,0,0
9,Le dire et dénoncer les effets collatéraux don...,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0
11,Ils nous envahissent avec des migrants musulma...,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0,0
14,C'est plus important de s'occuper des migrants...,0,0,0,0,0,0,0,1,1,0,0,1,0,0,1,0
15,J'aime les arbres.,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0
17,J'aime la France.,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0


In [13]:
df.to_csv("corpus_emotion_haineux_features.csv", index=True)